In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#descomprimir dataset para ETA
!unzip "/content/drive/MyDrive/Capstone/Datasets/mta_1706.csv.zip" -d "/content/drive/MyDrive/Capstone/Datasets/ETA/"

unzip:  cannot find or open /content/drive/MyDrive/Capstone/Datasets/mta_1706.csv.zip, /content/drive/MyDrive/Capstone/Datasets/mta_1706.csv.zip.zip or /content/drive/MyDrive/Capstone/Datasets/mta_1706.csv.zip.ZIP.


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
import pandas as pd

# La ruta a tu archivo .csv descomprimido
path_csv = "/content/drive/MyDrive/Capstone/Datasets/ETA/mta_1706.csv"

print("Cargando dataset...")

# on_bad_lines='skip' -> Ignora las filas que tienen un número incorrecto de columnas.
# low_memory=False -> Ayuda a evitar errores de tipo de datos en archivos grandes.
df = pd.read_csv(path_csv, on_bad_lines='skip', low_memory=False)

print("¡Dataset cargado exitosamente!")
print(df.head())

Cargando dataset...
¡Dataset cargado exitosamente!
        RecordedAtTime  DirectionRef PublishedLineName  \
0  2017-06-01 00:03:34             0                B8   
1  2017-06-01 00:03:43             1               S61   
2  2017-06-01 00:03:49             0              Bx10   
3  2017-06-01 00:03:31             0                Q5   
4  2017-06-01 00:03:22             1               Bx1   

                  OriginName  OriginLat  OriginLong  \
0                 4 AV/95 ST  40.616104  -74.031143   
1  ST GEORGE FERRY/S61 & S91  40.643169  -74.073494   
2     E 206 ST/BAINBRIDGE AV  40.875008  -73.880142   
3           TEARDROP/LAYOVER  40.701748  -73.802399   
4      RIVERDALE AV/W 231 ST  40.881187  -73.909340   

                   DestinationName  DestinationLat  DestinationLong  \
0          BROWNSVILLE ROCKAWAY AV       40.656048       -73.907379   
1                S I MALL YUKON AV       40.575935       -74.167686   
2                 RIVERDALE 263 ST       40.912376      

In [5]:
print("Tomando muestra de 3,000,000 filas...")
df_sample = df.sample(n=3000000, random_state=42)
print("Muestra tomada.")

Tomando muestra de 3,000,000 filas...
Muestra tomada.


In [ ]:
# --- 1. Limpiar filas con valores nulos (NaN) ---

print("Eliminando filas con valores nulos en columnas clave...")
# limpia filas que tengan valores nulos en estas 3 columnas
df_processed = df_sample.dropna(subset=['ExpectedArrivalTime', 'RecordedAtTime', 'DistanceFromStop']).copy()
print(f"Filas después de limpiar nulos: {len(df_processed)}")

Eliminando filas con valores nulos en columnas clave...
Filas después de limpiar nulos: 2610484


In [ ]:
# --- 2. Convertir columnas de tiempo ---
print("Convirtiendo columnas de tiempo...")
df_processed['ExpectedArrivalTime'] = pd.to_datetime(df_processed['ExpectedArrivalTime'])
df_processed['RecordedAtTime'] = pd.to_datetime(df_processed['RecordedAtTime'])

Convirtiendo columnas de tiempo...


In [ ]:
#imprimir para ver como quedaron las fechas/horas de las columnas
print(df_processed.head())

             RecordedAtTime  DirectionRef PublishedLineName  \
2170987 2017-06-10 16:09:29             0               B35   
5287764 2017-06-24 17:02:27             1           M15-SBS   
4304178 2017-06-20 14:45:31             0               Bx3   
1135968 2017-06-06 09:22:54             0               Q88   
1282111 2017-06-06 19:22:57             0                M7   

                         OriginName  OriginLat  OriginLong  \
2170987      CHURCH AV/MC DONALD AV  40.642651  -73.979584   
5287764               E 126 ST/2 AV  40.803230  -73.932449   
4304178             BROADWAY/179 ST  40.849327  -73.936508   
1135968                 92 ST/59 AV  40.734272  -73.869301   
1282111  AV OF THE AMERICAS/W 14 ST  40.737930  -73.996346   

                                 DestinationName  DestinationLat  \
2170987       BROWNSVILLE M GASTON BL via CHURCH       40.656345   
5287764  SELECT BUS SERVICE SOUTH FERRY via 2 AV       40.702122   
4304178                  RIVERDALE BWAY - 23

In [ ]:
# --- 3. Crear el Target (y) 'time_diff' ---
# Esta es la columna que queremos predecir (en minutos)
print("Creando el target (y) 'TimeToArrival' (en minutos)...")
df_processed['TimeToArrival'] = (df_processed['ExpectedArrivalTime'] - df_processed['RecordedAtTime']).dt.total_seconds()/60

Creando el target (y) 'TimeToArrival' (en minutos)...


In [ ]:
print(['TimeToArrival'].head())

AttributeError: 'list' object has no attribute 'head'

In [ ]:
# --- 4. Limpiar el Target
print("Limpiando el target...")

# Regla 1: Eliminar filas donde el bus ya pasó (tiempo negativo)
rows_before = len(df_processed)
df_processed = df_processed[df_processed['TimeToArrival'] > 0]
print(f"Filas eliminadas (tiempo negativo): {rows_before - len(df_processed)}")

Limpiando el target...
Filas eliminadas (tiempo negativo): 1347


In [ ]:
# Regla 2: Eliminar outliers (donde el bus está a más de 1 hora )
# El notebook asume que cualquier predicción mayor a 1 hora es un error o un outlier
rows_before = len(df_processed)
df_processed = df_processed[df_processed['TimeToArrival'] < 60]
print(f"Filas eliminadas (outliers > 1 hora): {rows_before - len(df_processed)}")

Filas eliminadas (outliers > 1 hora): 0


In [ ]:
# --- 5. Ingeniería de Características (Features) ---
# El notebook extrae todas las partes de la fecha
print("Creando features de tiempo (X)...")
df_processed['month'] = df_processed['RecordedAtTime'].dt.month
df_processed['day'] = df_processed['RecordedAtTime'].dt.day
df_processed['day_of_week'] = df_processed['RecordedAtTime'].dt.dayofweek # Lunes=0, Domingo=6
df_processed['hour'] = df_processed['RecordedAtTime'].dt.hour
df_processed['minute'] = df_processed['RecordedAtTime'].dt.minute

Creando features de tiempo (X)...


In [ ]:
# --- 6. Seleccionar Columnas Finales ---
print("Seleccionando columnas finales para el modelo...")
# Estas son las columnas que el notebook identifica como las features
features = ['DistanceFromStop', 'month', 'day', 'day_of_week', 'hour', 'minute']
target = 'TimeToArrival'

Seleccionando columnas finales para el modelo...


In [ ]:
# Creamos el DataFrame final listo para el entrenamiento
df_final = df_processed[features + [target]]

In [ ]:
# --- 7. Verificación Final ---
print("\n--- ¡Preprocesamiento completado! ---")
print(f"Total de filas listas para entrenar: {len(df_final)}")
print(df_final.head())


--- ¡Preprocesamiento completado! ---
Total de filas listas para entrenar: 2609137
         DistanceFromStop  month  day  day_of_week  hour  minute  \
2170987               2.0      6   10            5    16       9   
5287764             420.0      6   24            5    17       2   
4304178              79.0      6   20            1    14      45   
1135968              58.0      6    6            1     9      22   
1282111             113.0      6    6            1    19      22   

         TimeToArrival  
2170987       0.316667  
5287764       2.550000  
4304178       0.533333  
1135968       0.400000  
1282111       0.716667  


In [ ]:
#DATOS LIMPIOS GUARDADOS EN UN NUEVO CSV
print("Guardando el DataFrame preprocesado en un nuevo CSV...")

# Define la ruta de salida en tu Google Drive
ruta_salida = "/content/drive/MyDrive/Capstone/Datasets/mta_datos_limpios.csv"

# Guarda el df_final en un nuevo archivo CSV
# index=False es MUY importante para evitar que se guarde una columna extra
df_final.to_csv(ruta_salida, index=False)

print(f"¡Datos limpios guardados exitosamente en: {ruta_salida}")

Guardando el DataFrame preprocesado en un nuevo CSV...
¡Datos limpios guardados exitosamente en: /content/drive/MyDrive/Capstone/Datasets/mta_datos_limpios.csv


In [ ]:
#EJECUTAR A PARTIR DE ACA CADA VEZ QUE SE REINICIE EL ENTORNO
import pandas as pd

# 1. Montar Google Drive (como siempre)
from google.colab import drive
drive.mount('/content/drive')

# 2. Cargar directamente los datos LIMPIOS
print("Cargando datos preprocesados...")
ruta_limpia = "/content/drive/MyDrive/Capstone/Datasets/mta_datos_limpios.csv"
df_final = pd.read_csv(ruta_limpia)

print("¡Datos limpios listos para entrenar!")
print(df_final.head())

# 3. Ir DIRECTAMENTE a entrenar el modelo
# (Aquí pegas el código de train_test_split, RandomForestRegressor, etc.)
# ...

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando datos preprocesados...
¡Datos limpios listos para entrenar!
   DistanceFromStop  month  day  day_of_week  hour  minute  TimeToArrival
0               2.0      6   10            5    16       9       0.316667
1             420.0      6   24            5    17       2       2.550000
2              79.0      6   20            1    14      45       0.533333
3              58.0      6    6            1     9      22       0.400000
4             113.0      6    6            1    19      22       0.716667


In [ ]:
print("\nValores nulos restantes (deberían ser 0):")
print(df_final.isnull().sum())


Valores nulos restantes (deberían ser 0):
DistanceFromStop    0
month               0
day                 0
day_of_week         0
hour                0
minute              0
TimeToArrival       0
dtype: int64


In [ ]:
#ENTRENAMIENTO
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
# Definir X e y
X = df_final[features]
y = df_final[target]

In [ ]:
# Dividir en 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Entrenar el modelo
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15)
print("\nIniciando entrenamiento...")
rf_model.fit(X_train, y_train)
print("¡Modelo entrenado!")


Iniciando entrenamiento...
¡Modelo entrenado!


In [ ]:
# --- 1. Hacer Predicciones ---
# (Asegúrate de que 'rf_model' esté entrenado y que 'X_test' exista)
y_pred = rf_model.predict(X_test)

# --- 2. Imprimir Comparación (solo las primeras 10) ---
# Imprimir todo el bucle (range(len(y_pred))) puede colapsar tu navegador.
# Usamos .iloc[i] para acceder a y_test, ya que conserva su índice original.
print("--- Comparación de Predicción vs. Real (primeras 10) ---")
for i in range(10):
    print(f"Predicción: {y_pred[i]:.2f}   |  Valor Real: {y_test.iloc[i]:.2f} ")

# --- 3. Calcular Métricas de Evaluación ---
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)



--- Comparación de Predicción vs. Real (primeras 10) ---
Predicción: 0.40   |  Valor Real: 0.53 
Predicción: 0.27   |  Valor Real: 0.55 
Predicción: 0.32   |  Valor Real: 0.42 
Predicción: 1.06   |  Valor Real: 0.92 
Predicción: 1.29   |  Valor Real: 1.93 
Predicción: 0.79   |  Valor Real: 0.70 
Predicción: 0.27   |  Valor Real: 0.20 
Predicción: 0.38   |  Valor Real: 0.43 
Predicción: 0.82   |  Valor Real: 2.12 
Predicción: 0.30   |  Valor Real: 0.12 


In [ ]:
# --- 4. Imprimir Métricas ---
print("\n--- Métricas de Evaluación del Modelo ---")
print(f'Error Absoluto Medio (MAE):   {mae:.2f} minutos')
print(f'Error Cuadrático Medio (MSE): {mse:.2f}')
print(f'Raíz del ECM (RMSE):          {rmse:.2f} minutos')
print(f'Coeficiente R-cuadrado (R²):  {r2:.2f}')




--- Métricas de Evaluación del Modelo ---
Error Absoluto Medio (MAE):   0.35 minutos
Error Cuadrático Medio (MSE): 0.43
Raíz del ECM (RMSE):          0.66 minutos
Coeficiente R-cuadrado (R²):  0.84


In [ ]:
# --- 5. Guardar el Modelo (con Joblib) ---
# Usamos joblib para modelos de Scikit-learn
model_filename = 'rf_model_eta.pkl'
joblib.dump(rf_model, model_filename)

print(f"\n¡Modelo guardado exitosamente como '{model_filename}'!")


¡Modelo guardado exitosamente como 'rf_model_eta.pkl'!
